In [1]:
from datasets import load_from_disk

train_dataset = load_from_disk("../data/tokenized/train")
val_dataset = load_from_disk("../data/tokenized/val")

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})
Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 818
})


In [2]:
train_dataset[0]

{'id': '13818513',
 'dialogue': "Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.',
 'input_ids': [21603,
  10,
  21542,
  10,
  27,
  13635,
  5081,
  5,
  531,
  25,
  241,
  128,
  58,
  16637,
  10,
  10625,
  55,
  21542,
  10,
  27,
  31,
  195,
  830,
  25,
  5721,
  3,
  10,
  18,
  61,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]}

In [3]:
print(f"Length of input_ids (example 0) = {len(train_dataset[0]['input_ids'])}")
print(f"Length of labels (example 0) = {len(train_dataset[0]['labels'])}")

Length of input_ids (example 0) = 30
Length of labels (example 0) = 11


### Working with our Model

In [ ]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("t5-base")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

C:\Users\Yash pal\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Yash pal\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Yash pal\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system use

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

### Data Collator for Seq2Seq

In [6]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [7]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [9]:
# ROUGE during training

import numpy as np
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    # -100 cannot be decoded, so replace it with the pad token id
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for reference, prediction in zip(decoded_labels, decoded_preds):
        s = scorer.score(reference, prediction)
        for key in scores:
            scores[key].append(s[key].fmeasure)

    result = {key: round(float(np.mean(v)) * 100, 2) for key, v in scores.items()}
    result["gen_len"] = round(float(np.mean([np.count_nonzero(p != tokenizer.pad_token_id) for p in preds])), 1)

    return result

### Training the model

In [13]:
# Training Arguments

import math
import torch
from transformers import Seq2SeqTrainingArguments

EPOCHS = 5
BATCH_SIZE = 8       # if you get an out-of-memory error: BATCH_SIZE = 4 and ACCUM_STEPS = 4
ACCUM_STEPS = 2      # effective batch size = 8 x 2 = 16

steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * ACCUM_STEPS))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(0.05 * total_steps)

# bf16 only if the GPU supports it (fp16 often gives NaN losses with T5, so it is not used)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print(f"GPU: {torch.cuda.is_available()} | steps/epoch: {steps_per_epoch} | total: {total_steps} | warmup: {warmup_steps} | bf16: {use_bf16}")

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/Text Summarizer/results",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=ACCUM_STEPS,

    weight_decay=0.01,
    learning_rate=1e-4,
    lr_scheduler_type="linear",
    warmup_steps=warmup_steps,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=1,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    bf16=use_bf16,
    seed=42,
    report_to="none"
)

GPU: True | steps/epoch: 921 | total: 4605 | warmup: 230 | bf16: True


In [14]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [16]:
# Train the model
trainer.train(
    resume_from_checkpoint=True
)

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Gen Len
3,2.538231,1.367143,51.240000,26.510000,42.730000,24.400000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Gen Len
3,2.538231,1.367143,51.240000,26.510000,42.730000,24.400000
4,2.411535,1.373378,51.600000,27.260000,43.050000,24.900000
5,2.287529,1.380140,51.630000,27.090000,42.910000,24.700000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4605, training_loss=2.4351119970956057, metrics={'train_runtime': 5962.6349, 'train_samples_per_second': 7.412, 'train_steps_per_second': 0.463, 'total_flos': 2.872715555616768e+16, 'train_loss': 2.4351119970956057, 'epoch': 5.0})

In [17]:
# save model

model.save_pretrained(
    "/content/drive/MyDrive/Text Summarizer/models/saved_summary_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/Text Summarizer/models/saved_summary_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Text Summarizer/models/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/Text Summarizer/models/saved_summary_model/tokenizer.json')